<a href="https://colab.research.google.com/github/Aditya-Kumar-Sharma/MovieSuccessPrediction/blob/main/1_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Bollywood Box Office Predictor
## Notebook 1: Data Collection
**Goal:** Scrape YouTube trailer comments, music comments, and engagement metrics for 12 Bollywood films

### Pipeline Overview
```
YouTube API v3 → Evaluate limitations → Switch to youtube-comment-downloader
→ Scrape 2000+ comments per movie (trailer + music)
→ Scrape engagement metrics (views, likes, comments)
→ Save to Google Drive as CSV
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All files save here
SAVE_PATH = '/content/drive/MyDrive/Bollywood_Predictor/'

import os
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"✅ Save path ready: {SAVE_PATH}")

Mounted at /content/drive
✅ Save path ready: /content/drive/MyDrive/Bollywood_Predictor/


In [ ]:
!pip install youtube-comment-downloader google-api-python-client -q
print("✅ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 2.7 MB/s eta 0:00:00
✅ Libraries installed


In [ ]:
import pandas as pd
import numpy as np
import time
import json
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# YouTube comment downloader
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR

# YouTube Data API
from googleapiclient.discovery import build

print("✅ All imports successful")

✅ All imports successful


In [ ]:
# ── MASTER MOVIE DATABASE ─────────────────────────────────────────────────

movies = [
    {
        "name"              : "KGF Chapter 2",
        "year"              : 2022,
        "budget_cr"         : 100,
        "collection_cr"     : 1200,
        "genre"             : "Action",
        "language"          : "Pan-India",
        "is_sequel"         : 1,
        "is_remake"         : 0,
        "is_controversy"    : 0,
        "cast"              : ["Yash"],
        "director"          : "Prashanth Neel",
        "release_timing"    : "Holiday",
        "part1_collection"  : 250,
        "part1_verdict"     : "Hit",
        "franchise_gap_days": 1460,
        "same_cast"         : 1,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/JKa05nyUmuQ",
        "music_url_1"       : "https://youtu.be/suk3mW0tDPA",  # Mehabooba 225M
        "music_url_2"       : "https://youtu.be/zR5-HbFW6hc",  # Toofan 52M
        "music_views_1"     : 225_000_000,
        "music_views_2"     : 52_000_000,
    },
    {
        "name"              : "Jawan",
        "year"              : 2023,
        "budget_cr"         : 300,
        "collection_cr"     : 1100,
        "genre"             : "Action",
        "language"          : "Pan-India",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 0,
        "cast"              : ["Shah Rukh Khan"],
        "director"          : "Atlee",
        "release_timing"    : "Holiday",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/k8YiqM0Y-78",
        "music_url_1"       : "https://youtu.be/VAdGW7QDJiU",  # Chaleya 650M
        "music_url_2"       : "https://youtu.be/stjZKBhQ3lg",  # Zinda Banda 41M
        "music_views_1"     : 650_000_000,
        "music_views_2"     : 41_000_000,
    },
    {
        "name"              : "Pathaan",
        "year"              : 2023,
        "budget_cr"         : 250,
        "collection_cr"     : 1050,
        "genre"             : "Action",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 1,
        "cast"              : ["Shah Rukh Khan", "Deepika Padukone"],
        "director"          : "Siddharth Anand",
        "release_timing"    : "Holiday",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://www.youtube.com/watch?v=vqu4z34wENw",
        "music_url_1"       : "https://youtu.be/YxWlaYCA8MU",  # Jhoome Jo Pathaan 1.2B
        "music_url_2"       : "https://youtu.be/huxhqphtDrM",  # Beshram Rang 824M
        "music_views_1"     : 1_200_000_000,
        "music_views_2"     : 824_000_000,
    },
    {
        "name"              : "Animal",
        "year"              : 2023,
        "budget_cr"         : 200,
        "collection_cr"     : 900,
        "genre"             : "Action",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 1,
        "cast"              : ["Ranbir Kapoor"],
        "director"          : "Sandeep Reddy Vanga",
        "release_timing"    : "Normal",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/8FkLRUJj-o0",
        "music_url_1"       : "https://youtu.be/iAIBF2ngbWY",  # Pehle Bhi Main 410M
        "music_url_2"       : "https://youtu.be/zqGW6x_5N0k",  # Arjan Vailly 317M
        "music_views_1"     : 410_000_000,
        "music_views_2"     : 317_000_000,
    },
    {
        "name"              : "Pushpa 2",
        "year"              : 2024,
        "budget_cr"         : 500,
        "collection_cr"     : 1800,
        "genre"             : "Action",
        "language"          : "Pan-India",
        "is_sequel"         : 1,
        "is_remake"         : 0,
        "is_controversy"    : 0,
        "cast"              : ["Allu Arjun"],
        "director"          : "Sukumar",
        "release_timing"    : "Holiday",
        "part1_collection"  : 365,
        "part1_verdict"     : "Blockbuster",
        "franchise_gap_days": 1095,
        "same_cast"         : 1,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/1kVK0MZlbI4",
        "music_url_1"       : "https://youtu.be/MhIulWFPcpg",  # Pushpa Pushpa 149M
        "music_url_2"       : "https://youtu.be/0DVAM48BhQU",  # Angaaron 130M
        "music_views_1"     : 149_000_000,
        "music_views_2"     : 130_000_000,
    },
    {
        "name"              : "The Kashmir Files",
        "year"              : 2022,
        "budget_cr"         : 15,
        "collection_cr"     : 340,
        "genre"             : "Drama",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 1,
        "cast"              : ["Anupam Kher", "Mithun Chakraborty"],
        "director"          : "Vivek Agnihotri",
        "release_timing"    : "Normal",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/A179apttY58",
        "music_url_1"       : "https://youtu.be/C8QYVwX0M6g",  # Hum Dekhenge 6M
        "music_url_2"       : "https://youtu.be/C8QYVwX0M6g",  # same — only 1 song
        "music_views_1"     : 6_000_000,
        "music_views_2"     : 6_000_000,
    },
    {
        "name"              : "Tu Jhoothi Main Makkaar",
        "year"              : 2023,
        "budget_cr"         : 60,
        "collection_cr"     : 180,
        "genre"             : "RomCom",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 0,
        "cast"              : ["Ranbir Kapoor", "Shraddha Kapoor"],
        "director"          : "Luv Ranjan",
        "release_timing"    : "Holiday",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/Cx_Dtwn4ayw",
        "music_url_1"       : "https://youtu.be/2yeRRm9AIxI",  # Pyar Hota Kayi Baar 117M
        "music_url_2"       : "https://youtu.be/IMg_UUJVpMo",  # Tere Pyaar Mein 95M
        "music_views_1"     : 117_000_000,
        "music_views_2"     : 95_000_000,
    },
    {
        "name"              : "Rocky Aur Rani Ki Prem Kahani",
        "year"              : 2023,
        "budget_cr"         : 100,
        "collection_cr"     : 130,
        "genre"             : "RomCom",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 0,
        "cast"              : ["Ranveer Singh", "Alia Bhatt"],
        "director"          : "Karan Johar",
        "release_timing"    : "Normal",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/6mdxy3zohEk",
        "music_url_1"       : "https://youtu.be/QXJyMpxd210",  # Ve Kamleya 306M
        "music_url_2"       : "https://youtu.be/P1fIdFRnfqw",  # What Jhumka 126M
        "music_views_1"     : 306_000_000,
        "music_views_2"     : 126_000_000,
    },
    {
        "name"              : "Adipurush",
        "year"              : 2023,
        "budget_cr"         : 700,
        "collection_cr"     : 390,
        "genre"             : "Mythology",
        "language"          : "Pan-India",
        "is_sequel"         : 0,
        "is_remake"         : 0,
        "is_controversy"    : 1,
        "cast"              : ["Prabhas"],
        "director"          : "Om Raut",
        "release_timing"    : "Holiday",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/X7lRGozX8KQ",  # 2nd revised trailer
        "music_url_1"       : "https://youtu.be/Tl4bQBfOtbg",  # Ram Siya Ram 543M
        "music_url_2"       : "https://youtu.be/VJ-o8y8JqCQ",  # Jai Shree Ram 45M
        "music_views_1"     : 543_000_000,
        "music_views_2"     : 45_000_000,
    },
    {
        "name"              : "Bholaa",
        "year"              : 2023,
        "budget_cr"         : 120,
        "collection_cr"     : 60,
        "genre"             : "Action",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 1,
        "is_controversy"    : 0,
        "cast"              : ["Ajay Devgn"],
        "director"          : "Ajay Devgn",
        "release_timing"    : "Holiday",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "Blockbuster",
        "original_language" : "Tamil",
        "remake_type"       : "hindi_dubbed",   # ← add this
        "remake_novelty"    : 0.3,
        "trailer_url"       : "https://youtu.be/K-EMszLvRIQ",
        "music_url_1"       : "https://youtu.be/n5ZKnHJmyrU",  # Najar Lag Jayegi 52M
        "music_url_2"       : "https://youtu.be/jcsiMH3Go8I",  # Pan Dukaniya 3.7M
        "music_views_1"     : 52_000_000,
        "music_views_2"     : 3_700_000,
    },
    {
        "name"              : "Selfiee",
        "year"              : 2023,
        "budget_cr"         : 100,
        "collection_cr"     : 45,
        "genre"             : "Comedy",
        "language"          : "Hindi",
        "is_sequel"         : 0,
        "is_remake"         : 1,
        "is_controversy"    : 0,
        "cast"              : ["Akshay Kumar"],
        "director"          : "Raj Mehta",
        "release_timing"    : "Normal",
        "part1_collection"  : 0,
        "part1_verdict"     : "NA",
        "franchise_gap_days": 0,
        "same_cast"         : 0,
        "original_verdict"  : "Blockbuster",
        "original_language" : "Malayalam",
        "remake_type"       : "regional_only",  # ← add this
        "remake_novelty"    : 0.8,
        "trailer_url"       : "https://youtu.be/lS1KScfdr70",
        "music_url_1"       : "https://youtu.be/ha3QcOXcUyY",  # Main Khiladi 205M
        "music_url_2"       : "https://youtu.be/eizIc5eQiEM",  # Selfiee Title Track
        "music_views_1"     : 205_000_000,
        "music_views_2"     : 0,  # update with actual views
    },
    {
        "name"              : "OMG 2",
        "year"              : 2023,
        "budget_cr"         : 50,
        "collection_cr"     : 180,
        "genre"             : "Drama",
        "language"          : "Hindi",
        "is_sequel"         : 1,
        "is_remake"         : 0,
        "is_controversy"    : 1,
        "cast"              : ["Akshay Kumar", "Pankaj Tripathi"],
        "director"          : "Amit Rai",
        "release_timing"    : "Normal",
        "part1_collection"  : 120,
        "part1_verdict"     : "Hit",
        "franchise_gap_days": 3650,
        "same_cast"         : 0,  # different lead cast
        "original_verdict"  : "NA",
        "original_language" : "NA",
        "remake_type"       : "none",
        "remake_novelty"    : 1.0,
        "trailer_url"       : "https://youtu.be/Y6ZKXqM7HNQ",
        "music_url_1"       : "https://youtu.be/YT8rY_o5VhY",  # OOnchi OOnchi 112M
        "music_url_2"       : "https://youtu.be/TrVaKSCUttU",  # Har Har Mahadev 64M
        "music_views_1"     : 112_000_000,
        "music_views_2"     : 64_000_000,
    },
]

# ── Calculate ROI and print summary ──────────────────────────────────────
print(f"✅ Movie database loaded: {len(movies)} movies")
print(f"\n{'Movie':<40} {'ROI':>8}  {'Label':<15} {'Music1':>12} {'Music2':>12}")
print("─" * 90)

for m in movies:
    roi   = ((m['collection_cr'] - m['budget_cr']) / m['budget_cr']) * 100
    label = ('🌟 Blockbuster' if roi > 150 else
             '✅ Hit'         if roi > 50  else
             '😐 Average'    if roi > 0   else
             '❌ Flop')
    m1v   = f"{m['music_views_1']/1e6:.0f}M"
    m2v   = f"{m['music_views_2']/1e6:.0f}M"
    print(f"{m['name']:<40} {roi:>7.0f}%  {label:<15} {m1v:>12} {m2v:>12}")

# ── Music views classification ────────────────────────────────────────────
def classify_music_reach(views):
    if views   >= 500_000_000: return 'Mega Viral'
    elif views >= 100_000_000: return 'Viral'
    elif views >= 50_000_000 : return 'Strong'
    elif views >= 10_000_000 : return 'Moderate'
    else                     : return 'Low'

print(f"\n{'Movie':<40} {'Top Song':<12} {'Reach':<12} {'Consistency'}")
print("─" * 80)
for m in movies:
    v1    = m['music_views_1']
    v2    = m['music_views_2']
    reach = classify_music_reach(v1)
    consistency = 'Both Strong' if v2 >= 50_000_000 else 'Drop-off'
    print(f"{m['name']:<40} {v1/1e6:>8.0f}M   {reach:<12} {consistency}")

✅ Movie database loaded: 12 movies

Movie                                         ROI  Label                 Music1       Music2
──────────────────────────────────────────────────────────────────────────────────────────
KGF Chapter 2                               1100%  🌟 Blockbuster           225M          52M
Jawan                                        267%  🌟 Blockbuster           650M          41M
Pathaan                                      320%  🌟 Blockbuster          1200M         824M
Animal                                       350%  🌟 Blockbuster           410M         317M
Pushpa 2                                     260%  🌟 Blockbuster           149M         130M
The Kashmir Files                           2167%  🌟 Blockbuster             6M           6M
Tu Jhoothi Main Makkaar                      200%  🌟 Blockbuster           117M          95M
Rocky Aur Rani Ki Prem Kahani                 30%  😐 Average               306M         126M
Adipurush                           

In [ ]:
# ── YOUTUBE API v3 EVALUATION ──
# Demonstrating why official API is insufficient for large-scale comment analysis

print("=" * 50)
print("📊 YOUTUBE DATA API v3 — LIMITATION ANALYSIS")
print("=" * 50)

print("""
Official YouTube Data API v3 Facts:
──────────────────────────────────────────────────
✅ Pros:
   • Official Google product — reliable and structured
   • Returns comment metadata (likes, replies, timestamp)
   • Supports filtering and sorting

❌ Limitations for our use case:
   • Free quota: 10,000 units/day
   • commentThreads.list = 1 unit per call
   • Max 100 comments per API call
   • To get 2,000 comments = 20 API calls
   • For 12 movies × 2 sources = 24 URLs × 20 calls = 480 calls
   • Each video's comments may need pagination (nextPageToken)
   • Real-time quota exhausts quickly — no bulk scraping
   • Requires API key setup and OAuth for some endpoints

📊 Our requirement:
   • 12 movies × 2 URLs (trailer + music) = 24 sources
   • Target: 2,000 comments per source
   • Total: ~48,000 comments needed
   • API would exhaust daily quota in first 2-3 movies

✅ Decision: Use youtube-comment-downloader
   • No API key required
   • No rate limits
   • Unlimited comments
   • Same comment data structure
   • Industry acceptable for research/academic projects
""")

print("=" * 50)
print("✅ Proceeding with youtube-comment-downloader")
print("=" * 50)

📊 YOUTUBE DATA API v3 — LIMITATION ANALYSIS

Official YouTube Data API v3 Facts:
──────────────────────────────────────────────────
✅ Pros:
   • Official Google product — reliable and structured
   • Returns comment metadata (likes, replies, timestamp)
   • Supports filtering and sorting

❌ Limitations for our use case:
   • Free quota: 10,000 units/day
   • commentThreads.list = 1 unit per call
   • Max 100 comments per API call
   • To get 2,000 comments = 20 API calls
   • For 12 movies × 2 sources = 24 URLs × 20 calls = 480 calls
   • Each video's comments may need pagination (nextPageToken)
   • Real-time quota exhausts quickly — no bulk scraping
   • Requires API key setup and OAuth for some endpoints

📊 Our requirement:
   • 12 movies × 2 URLs (trailer + music) = 24 sources
   • Target: 2,000 comments per source
   • Total: ~48,000 comments needed
   • API would exhaust daily quota in first 2-3 movies

✅ Decision: Use youtube-comment-downloader
   • No API key required
   • No r

In [ ]:
#  COMMENT SCRAPING FUNCTION

def scrape_comments(url, max_comments=2000, source_type="trailer"):
    downloader = YoutubeCommentDownloader()
    comments   = []

    try:
        generator = downloader.get_comments_from_url(
            url,
            sort_by=SORT_BY_POPULAR
        )

        for i, comment in enumerate(generator):
            if i >= max_comments:
                break

            comments.append({
                'text'       : comment.get('text', ''),
                'likes'      : comment.get('votes', 0),
                'reply_count': comment.get('reply_count', 0),
                'time'       : comment.get('time', ''),
                'source_type': source_type,
            })

            # Progress update every 500 comments
            if (i + 1) % 500 == 0:
                print(f"      → {i+1} comments scraped...")

    except Exception as e:
        print(f"      ⚠️  Error: {e}")

    return comments

print("✅ Scraping function defined")

✅ Scraping function defined


In [ ]:
#  ENGAGEMENT METRICS FUNCTION
def get_video_metrics(url):
    import subprocess
    import json

    # Extract video ID from URL
    if 'v=' in url:
        video_id = url.split('v=')[1].split('&')[0]
    elif 'youtu.be/' in url:
        video_id = url.split('youtu.be/')[1].split('?')[0]
    else:
        video_id = url

    try:
        # Use yt-dlp to get video metadata
        result = subprocess.run(
            ['yt-dlp', '--dump-json', '--no-download',
             f'https://www.youtube.com/watch?v={video_id}'],
            capture_output=True, text=True, timeout=30
        )

        if result.returncode == 0:
            data = json.loads(result.stdout)
            return {
                'view_count'    : data.get('view_count', 0),
                'like_count'    : data.get('like_count', 0),
                'comment_count' : data.get('comment_count', 0),
                'duration'      : data.get('duration', 0),
                'upload_date'   : data.get('upload_date', ''),
                'title'         : data.get('title', ''),
            }
    except Exception as e:
        print(f"      ⚠️  Metrics error: {e}")

    return {
        'view_count': 0, 'like_count': 0,
        'comment_count': 0, 'duration': 0,
        'upload_date': '', 'title': ''
    }

# Install yt-dlp
import subprocess
subprocess.run(['pip', 'install', 'yt-dlp', '-q'])
print("✅ Metrics function defined")

✅ Metrics function defined


In [ ]:
# ── MAIN SCRAPING LOOP ────────────────────────────────────────────────────

all_comments    = []
trailer_metrics = []
music_metrics   = []

print("🎬 Starting data collection for all movies...")
print("=" * 60)

for idx, movie in enumerate(movies):
    print(f"\n[{idx+1}/{len(movies)}] 🎥 {movie['name']}")
    print("-" * 50)

    # ── Trailer Comments
    print(f"   📹 Scraping trailer comments...")
    t_comments = scrape_comments(
        movie['trailer_url'],
        max_comments=2000,
        source_type='trailer'
    )
    for c in t_comments:
        c['movie']  = movie['name']
        c['source'] = 'trailer'
    all_comments.extend(t_comments)
    print(f"   ✅ Trailer: {len(t_comments)} comments")

    # ── Trailer Metrics
    print(f"   📊 Fetching trailer metrics...")
    t_metrics           = get_video_metrics(movie['trailer_url'])
    t_metrics['movie']  = movie['name']
    t_metrics['source'] = 'trailer'
    trailer_metrics.append(t_metrics)
    print(f"   ✅ Views: {t_metrics['view_count']:,}")

    time.sleep(2)

    # ── Music Song 1 Comments
    print(f"   🎵 Scraping music song 1 comments...")
    m1_comments = scrape_comments(
        movie['music_url_1'],
        max_comments=1000,
        source_type='music_1'
    )
    for c in m1_comments:
        c['movie']  = movie['name']
        c['source'] = 'music_1'
    all_comments.extend(m1_comments)
    print(f"   ✅ Music 1: {len(m1_comments)} comments")

    # ── Music Song 1 Metrics
    m1_metrics              = get_video_metrics(movie['music_url_1'])
    m1_metrics['movie']     = movie['name']
    m1_metrics['source']    = 'music_1'
    m1_metrics['manual_views'] = movie['music_views_1']
    music_metrics.append(m1_metrics)

    time.sleep(2)

    # ── Music Song 2 Comments
    print(f"   🎵 Scraping music song 2 comments...")

    # Skip if same URL as song 1 (Kashmir Files case)
    if movie['music_url_2'] != movie['music_url_1']:
        m2_comments = scrape_comments(
            movie['music_url_2'],
            max_comments=1000,
            source_type='music_2'
        )
        for c in m2_comments:
            c['movie']  = movie['name']
            c['source'] = 'music_2'
        all_comments.extend(m2_comments)
        print(f"   ✅ Music 2: {len(m2_comments)} comments")

        # ── Music Song 2 Metrics
        m2_metrics              = get_video_metrics(movie['music_url_2'])
        m2_metrics['movie']     = movie['name']
        m2_metrics['source']    = 'music_2'
        m2_metrics['manual_views'] = movie['music_views_2']
        music_metrics.append(m2_metrics)
    else:
        print(f"   ⏭️  Music 2 same as Music 1 — skipping")

    time.sleep(3)

print("\n" + "=" * 60)
print(f"🎉 Scraping complete!")
print(f"   Total comments collected : {len(all_comments):,}")
print(f"   Movies processed         : {len(trailer_metrics)}")
print(f"\n📊 Comments breakdown:")
comments_temp = {}
for c in all_comments:
    key = f"{c['movie']} ({c['source']})"
    comments_temp[key] = comments_temp.get(key, 0) + 1
for k, v in sorted(comments_temp.items()):
    print(f"   {k:<55} : {v:,}")

🎬 Starting data collection for all movies...

[1/12] 🎥 KGF Chapter 2
--------------------------------------------------
   📹 Scraping trailer comments...
      → 500 comments scraped...
      → 1000 comments scraped...
      → 1500 comments scraped...
      → 2000 comments scraped...
   ✅ Trailer: 2000 comments
   📊 Fetching trailer metrics...
   ✅ Views: 125,839,413
   🎵 Scraping music song 1 comments...
      → 500 comments scraped...
      → 1000 comments scraped...
   ✅ Music 1: 1000 comments
   🎵 Scraping music song 2 comments...
      → 500 comments scraped...
      → 1000 comments scraped...
   ✅ Music 2: 1000 comments

[2/12] 🎥 Jawan
--------------------------------------------------
   📹 Scraping trailer comments...
      → 500 comments scraped...
      → 1000 comments scraped...
      → 1500 comments scraped...
      → 2000 comments scraped...
   ✅ Trailer: 2000 comments
   📊 Fetching trailer metrics...
   ✅ Views: 87,607,904
   🎵 Scraping music song 1 comments...
      → 500

In [ ]:
# ── SAVE ALL DATA TO GOOGLE DRIVE ─────────────────────────────────────────

# Comments DataFrame
comments_df = pd.DataFrame(all_comments)
comments_df.to_csv(f'{SAVE_PATH}comments_raw.csv', index=False)
print(f"✅ Comments saved: {comments_df.shape}")

# Trailer metrics DataFrame
trailer_df = pd.DataFrame(trailer_metrics)
trailer_df.to_csv(f'{SAVE_PATH}trailer_metrics.csv', index=False)
print(f"✅ Trailer metrics saved: {trailer_df.shape}")

# Music metrics DataFrame
music_df = pd.DataFrame(music_metrics)
music_df.to_csv(f'{SAVE_PATH}music_metrics.csv', index=False)
print(f"✅ Music metrics saved: {music_df.shape}")

# Movies master info
movies_df = pd.DataFrame(movies)
movies_df['cast'] = movies_df['cast'].apply(lambda x: ', '.join(x))
movies_df['roi_pct'] = ((movies_df['collection_cr'] - movies_df['budget_cr'])
                         / movies_df['budget_cr'] * 100).round(2)
movies_df['label'] = movies_df['roi_pct'].apply(
    lambda x: 'Blockbuster' if x > 150 else
              'Hit'         if x > 50  else
              'Average'     if x > 0   else 'Flop'
)
movies_df.to_csv(f'{SAVE_PATH}movies_master.csv', index=False)
print(f"✅ Movies master saved: {movies_df.shape}")

# Summary
print(f"\n📊 Collection Summary:")
print(comments_df.groupby(['movie','source'])['text'].count().unstack())

✅ Comments saved: (46696, 7)
✅ Trailer metrics saved: (12, 8)
✅ Music metrics saved: (23, 9)
✅ Movies master saved: (12, 27)

📊 Collection Summary:
source                         music_1  music_2  trailer
movie                                                   
Adipurush                       1000.0   1000.0   2000.0
Animal                          1000.0   1000.0   2000.0
Bholaa                          1000.0    696.0   2000.0
Jawan                           1000.0   1000.0   2000.0
KGF Chapter 2                   1000.0   1000.0   2000.0
OMG 2                           1000.0   1000.0   2000.0
Pathaan                         1000.0   1000.0   2000.0
Pushpa 2                        1000.0   1000.0   2000.0
Rocky Aur Rani Ki Prem Kahani   1000.0   1000.0   2000.0
Selfiee                         1000.0   1000.0   2000.0
The Kashmir Files               1000.0      NaN   2000.0
Tu Jhoothi Main Makkaar         1000.0   1000.0   2000.0


In [ ]:
# ── DEBUG: TEST EACH URL DIRECTLY ─────────────────────────────────────────

from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR

def test_url(url, movie_name):
    downloader = YoutubeCommentDownloader()
    try:
        generator = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
        # Try to get just first comment
        first = next(generator, None)
        if first:
            print(f"✅ {movie_name} → WORKS — sample: {str(first.get('text',''))[:50]}")
        else:
            print(f"❌ {movie_name} → Empty generator — comments disabled or wrong URL")
    except Exception as e:
        print(f"⚠️  {movie_name} → Error: {e}")

print("🔍 Testing all trailer URLs...\n")
for movie in movies:
    test_url(movie['trailer_url'], movie['name'])

🔍 Testing all trailer URLs...

✅ KGF Chapter 2 → WORKS — sample: Ab kuch log aayenge aur bolenge who come after tox
✅ Jawan → WORKS — sample: Too good🥵🔥🥵🔥
✅ Pathaan → WORKS — sample: Experience the world of #YRFSpyUniverse, watch all
✅ Animal → WORKS — sample: THE ANIMAL, YOU HAVE NEVER SEEN BEFORE! 🔥
✅ Pushpa 2 → WORKS — sample: Fire Nahi, Wild Fire Hai Apna Pushpa🔥🔥
✅ The Kashmir Files → WORKS — sample: Who are here after pahalgam attack ?
✅ Tu Jhoothi Main Makkaar → WORKS — sample: Ranbir Kapoor + Shraddha Kapoor = Entertainment Gu
✅ Rocky Aur Rani Ki Prem Kahani → WORKS — sample: After Dhurandhar😂😅
✅ Adipurush → WORKS — sample: Jai Shri Ram 🚩🚩🚩🚩
✅ Bholaa → WORKS — sample: Ladaiyaan hauslon se jeeti jaati hai, sankhyan, ba
✅ Selfiee → WORKS — sample: It's great to see emraan hashmi sharing screen spa
✅ OMG 2 → WORKS — sample: 2:00 Ganga Kha Se Bheti H Mujhe Mt Btana That was 
